# Langkah 9 — Analisis Statistik & Data Dashboard

**Framing penting:** dataset = ulasan **1–2 bintang saja**. Ini BUKAN skor kepuasan, melainkan
**profil keluhan** — "ini masukan-masukan dari warga". Justifikasi metodologis: ulasan negatif
memuat umpan balik paling **spesifik & actionable** untuk perbaikan layanan (ulasan positif
cenderung afirmatif-generik). Sejajar dengan *complaint mining* — sengaja fokus sinyal negatif
karena itulah yang menggerakkan kebijakan dinas.

Karena hampir semua ulasan negatif, rasio negatif absolut tidak informatif. Yang bermakna:
- **Komposisi keluhan** — dari semua keluhan di puskesmas itu, porsi tiap dimensi (apa masalah utamanya).
- **Intensitas relatif** — seberapa sering dimensi dikeluhkan per ulasan, dibanding peer (uji 2-proporsi).
- **Skor 1–5** = persentil intensitas (ter-*shrink* Bayesian) → **5 = paling banyak dikeluhkan**.
  Arah skala bisa dibalik kapan saja (parameter `higher_is_worse`).

Empat komponen statistik: (1) skor keluhan + Wilson CI + EB shrinkage, (2) badge uji 2-proporsi
vs peer + mismatch bintang-teks, (3) χ² antar-wilayah + residual + lift co-occurrence,
(4) kanonikalisasi sub_issue (clustering).

In [1]:
import sys, json
from pathlib import Path

ROOT = Path('..').resolve()
DATA_DIR = ROOT.parent / 'data'
OUT = ROOT / 'outputs'
sys.path.insert(0, str(ROOT))

import importlib
import pandas as pd
import numpy as np
import absa.stats
importlib.reload(absa.stats)
from absa import stats
from absa.prompts import SERVQUAL_DIMS

findings = pd.read_csv(OUT / 'findings_full.csv', encoding='utf-8-sig')
findings['review_id'] = findings['review_id'].astype(str)
reviews = pd.read_csv(DATA_DIR / 'reviews_cleaned_rating_1_2.csv')
reviews['review_id'] = reviews['review_id'].astype(str)

print(f'Temuan: {len(findings)} | Ulasan dgn temuan: {findings["review_id"].nunique()}')
print(f'Puskesmas: {findings["puskesmas_id"].nunique()} | Wilayah: {findings["wilayah"].nunique()}')

Temuan: 17375 | Ulasan dgn temuan: 8267
Puskesmas: 127 | Wilayah: 3


## 1. Skor keluhan per puskesmas

**Apa yang dihitung:** untuk tiap puskesmas × dimensi, dua angka — **komposisi** (dari semua
keluhan puskesmas itu, berapa persen jatuh di dimensi ini → apa *masalah utamanya*) dan
**intensitas** (dari semua ulasannya, berapa porsi yang menyebut dimensi ini sebagai keluhan).

**Kenapa pakai Wilson CI + Bayesian shrinkage, bukan persen mentah:**
- Persen mentah menipu untuk puskesmas ber-ulasan sedikit. Contoh: 2 ulasan, 2-duanya mengeluh
  → 100%. Tapi itu cuma 2 orang; kita tidak boleh seyakin itu. **Wilson CI** memberi selang
  ketidakpastian, dan **shrinkage empirical-Bayes** *menarik* angka puskesmas kecil mendekati
  rata-rata kabupaten (makin sedikit data, makin ditarik) — jadi puskesmas tidak di-ranking
  berdasarkan kebetulan.
- **Skor 1–5** = persentil intensitas (yang sudah di-shrink) di antara semua puskesmas. Dipakai
  persentil karena hampir semua ulasan negatif → angka absolut menumpuk; yang informatif adalah
  *posisi relatif*. `HIGHER_IS_WORSE=True` → 5 = paling banyak dikeluhkan (bisa dibalik).

**Kontribusi ke dashboard:** angka "skor per dimensi" + badge **"Cukup banyak data / Datanya
sedang / Data terbatas"** (dari ukuran sampel) di tab Profil Puskesmas.

In [ ]:
HIGHER_IS_WORSE = True   # True: 5 = paling banyak dikeluhkan. Balik ke False kapan saja.

skor = stats.complaint_scores(findings, min_n_score=10, higher_is_worse=HIGHER_IS_WORSE)
print(f'Baris skor (puskesmas x dimensi): {len(skor)}')
print('Sebaran skor 1-5 (ada variasi nyata):')
print(skor['skor_1_5'].describe().round(2).to_string())

print('\nContoh — satu puskesmas (>=30 ulasan):')
pid = skor[skor['n_reviews']>=30]['puskesmas_id'].iloc[0]
cols = ['dimension','n_reviews','komposisi_pct','intensitas_rate','intensitas_ci95','skor_1_5','kepercayaan']
print(skor[skor['puskesmas_id']==pid][cols].to_string(index=False))

print('\nEfek shrinkage (n kecil ditarik lebih kuat ke rata-rata):')
demo = skor.copy()
demo['selisih'] = (demo['intensitas_shrunk'] - demo['intensitas_rate']).abs()
print(demo.groupby(pd.cut(demo['n_reviews'], [0,5,10,30,9999]), observed=True)['selisih'].mean().round(3).to_string())

## 2. Badge perbandingan — puskesmas vs puskesmas lain (uji dua proporsi)

**Apa:** menguji apakah satu puskesmas dikeluhkan soal dimensi tertentu **lebih sering / lebih
jarang** daripada gabungan semua puskesmas lain (peer-nya).

**Kenapa pakai uji hipotesis (uji dua proporsi z), bukan sekadar bandingkan angka:** dua persen
yang beda (mis. 40% vs 35%) belum tentu beda *sungguhan* — bisa cuma noise sampel. Uji z menjawab:
"apakah selisih ini cukup besar untuk dipercaya, bukan kebetulan?" Hanya yang signifikan
(p<0.05) yang diberi label; sisanya 'setara'. Ini mengubah klaim ranking jadi klaim yang teruji.

**`mismatch bintang-teks`:** menandai puskesmas berbintang tinggi tapi teksnya banyak keluhan.
*(Pada data ini selalu kosong karena hanya ada ulasan 1–2 bintang — baru aktif kalau ulasan
positif diproses.)*

**Kontribusi:** badge **"lebih sering dikeluhkan dari puskesmas sejenis"** di kartu dimensi.

In [3]:
peer = stats.peer_comparison(skor, alpha=0.05)
print('Verdict vs peer (signifikan pada α=0.05):')
print(peer['vs_peer'].value_counts().to_string())

mismatch = stats.rating_text_mismatch(findings, reviews)
n_mis = mismatch['mismatch'].sum()
print(f'\nPuskesmas "bintang tinggi tapi banyak keluhan teks": {n_mis}')
if n_mis:
    print(mismatch[mismatch['mismatch']].head(10)[['puskesmas_id','rating_bintang','text_neg_rate','n_reviews']].to_string(index=False))

Verdict vs peer (signifikan pada α=0.05):
vs_peer
setara                     486
lebih jarang dikeluhkan     66
lebih sering dikeluhkan     65

Puskesmas "bintang tinggi tapi banyak keluhan teks": 0


## 3. Apakah pola keluhan sama di tiap wilayah? (uji χ²) + keluhan yang datang bersamaan (lift)

**Uji χ² independensi — apa & kenapa:** menguji apakah *komposisi* keluhan berbeda antar wilayah
(Bantul/Semarang/Surabaya). χ² cocok karena datanya kategorikal (dimensi × wilayah, berupa
hitungan). **Residual terstandarisasi** menunjukkan sel mana yang menyimpang: nilai |>2| berarti
wilayah itu mengeluh dimensi itu jauh lebih/ kurang dari yang diharapkan bila pola sama di mana-mana.

**Lift co-occurrence — apa & kenapa:** mengukur dimensi mana yang sering muncul **bersamaan dalam
satu ulasan**. `lift = P(A&B) / (P(A)·P(B))`; >1 = lebih sering bareng dari kebetulan. Berguna
untuk insight "warga yang mengeluh X biasanya juga mengeluh Y → benahi bersamaan".

**Kontribusi:** tabel **"Apakah sama di tiap wilayah?"** + kotak **"Keluhan yang sering datang
bersamaan"** di tab Topik Keluhan.

In [4]:
chi = stats.region_dimension_chi2(findings)
print(f"χ² = {chi['chi2']}, dof = {chi['dof']}, p = {chi['p']:.2e}")
print('=> distribusi keluhan', 'BERBEDA' if chi['p']<0.05 else 'TIDAK berbeda', 'signifikan antar wilayah.')
print('\nTabel kontingensi (temuan negatif, dimensi x wilayah):')
print(chi['contingency'].to_string())
print('\nResidual terstandarisasi (|>2| = sel menyimpang dari independensi):')
print(chi['std_residuals'].to_string())

χ² = 101.258, dof = 8, p = 2.36e-18
=> distribusi keluhan BERBEDA signifikan antar wilayah.

Tabel kontingensi (temuan negatif, dimensi x wilayah):
wilayah         Bantul  Semarang  Surabaya
dimension                                 
Assurance          260       260      1465
Empathy            842       592      3932
Reliability        633       705      3222
Responsiveness     650       388      2670
Tangibles           98        95       341

Residual terstandarisasi (|>2| = sel menyimpang dari independensi):
wilayah         Bantul  Semarang  Surabaya
dimension                                 
Assurance        -2.58      0.59      0.95
Empathy           0.60     -3.29      1.10
Reliability      -2.57      5.38     -1.07
Responsiveness    3.35     -3.71      0.01
Tangibles         1.76      3.36     -2.22


In [5]:
lift = stats.cooccurrence_lift(findings)
print('Co-occurrence dimensi dalam satu ulasan (lift > 1 = lebih sering bersamaan dari kebetulan):')
print(lift.to_string(index=False))

Co-occurrence dimensi dalam satu ulasan (lift > 1 = lebih sering bersamaan dari kebetulan):
         dim_a       dim_b   p_a   p_b  p_both  lift
     Assurance     Empathy 0.207 0.551   0.125  1.10
Responsiveness Reliability 0.425 0.472   0.212  1.06
   Reliability   Assurance 0.472 0.207   0.094  0.96
Responsiveness   Tangibles 0.425 0.068   0.028  0.95
       Empathy   Tangibles 0.551 0.068   0.031  0.81
     Assurance   Tangibles 0.207 0.068   0.011  0.81
   Reliability   Tangibles 0.472 0.068   0.025  0.79
   Reliability     Empathy 0.472 0.551   0.205  0.79
Responsiveness     Empathy 0.425 0.551   0.162  0.69
Responsiveness   Assurance 0.425 0.207   0.056  0.64


## 4. Menyatukan ribuan frasa keluhan jadi sedikit isu (clustering semantik)

**Masalah:** model menulis `sub_issue` sebagai teks bebas → ribuan frasa unik ("antre lama",
"nunggu lama", "antre berjam-jam" semuanya MASALAH yang sama). Tanpa disatukan, hitungan
"Antre lama — 38 ulasan" mustahil.

**Kenapa embedding semantik, bukan kemiripan ejaan:** mencocokkan ejaan gagal — "antre lama" &
"nunggu lama" beda huruf tapi sama makna (harus digabung), sedangkan "antre lama" & "antre obat
lama" mirip huruf tapi beda masalah. **Embedding multibahasa** memetakan frasa ke vektor
*makna*, lalu **KMeans** mengelompokkannya; jumlah kelompok dipilih otomatis lewat **silhouette
score** (mengukur seberapa rapi clusternya). Nama tiap isu kanonik = frasa paling sering di
kelompoknya.

**Kontribusi:** daftar **"sumber komplain"** ("Antre lama — N ulasan") + contoh kutipan asli
di tiap kartu dimensi. *(Perlu paket `sentence-transformers`.)*

In [ ]:
# Kanonikalisasi sub_issue pakai EMBEDDING SEMANTIK (dari absa.stats).
# Mengganti pendekatan ejaan lama yang runtuh jadi 1 cluster.
demo = stats.canonicalize_sub_issues(findings, 'Responsiveness', k_range=(6, 12))
print(f"Responsiveness: {demo['sub_issue'].nunique()} sub_issue mentah -> {demo['cluster_label'].nunique()} isu kanonik")
print('\nTop isu kanonik (jumlah ulasan):')
print(demo.groupby('cluster_label')['review_id'].nunique().sort_values(ascending=False).head(12).to_string())

## 5. Rakit data dashboard → JSON

Dua file:
- `puskesmas_profiles.json` — per puskesmas: skor tiap dimensi (1–5, CI, kepercayaan),
  verdict vs peer, top sub_issue kanonik + jumlah ulasan, contoh quote.
- `topik_kabupaten.json` — sebaran tiap topik, χ² antar-wilayah, co-occurrence.

In [ ]:
# Pra-hitung sub_issue kanonik (embedding semantik) + contoh quote untuk semua dimensi
canon_by_dim = {d: stats.canonicalize_sub_issues(findings, d, k_range=(6, 12)) for d in SERVQUAL_DIMS}

def top_sub_issues(pid, dim, k=3):
    c = canon_by_dim[dim]
    c = c[c['puskesmas_id']==pid]
    if c.empty: return []
    g = c.groupby('cluster_label').agg(n=('review_id','nunique'),
                                       quote=('quote','first')).sort_values('n', ascending=False)
    return [{'isu': idx, 'n_ulasan': int(r['n']), 'contoh_quote': str(r['quote'])[:200]}
            for idx, r in g.head(k).iterrows()]

peer_lookup = {(r['puskesmas_id'], r['dimension']): r['vs_peer'] for _, r in peer.iterrows()}
mismatch_lookup = {r['puskesmas_id']: bool(r['mismatch']) for _, r in mismatch.iterrows()}

profiles = {}
for pid, grp in skor.groupby('puskesmas_id'):
    pname = grp['puskesmas_name'].iloc[0]; wil = grp['wilayah'].iloc[0]
    dims = {}
    for _, r in grp.iterrows():
        d = r['dimension']
        dims[d] = {
            'skor_1_5': r['skor_1_5'],
            'komposisi_pct': r['komposisi_pct'],
            'intensitas_rate': r['intensitas_rate'],
            'intensitas_ci95': r['intensitas_ci95'],
            'n_reviews': int(r['n_reviews']),
            'kepercayaan': r['kepercayaan'],
            'cukup_dinilai': bool(r['cukup_dinilai']),
            'vs_peer': peer_lookup.get((pid, d), 'setara'),
            'top_sub_issues': top_sub_issues(pid, d),
        }
    profiles[pid] = {
        'nama': pname, 'wilayah': wil,
        'mismatch_bintang_teks': mismatch_lookup.get(pid, False),
        'arah_skor': 'tinggi = paling banyak dikeluhkan' if HIGHER_IS_WORSE else 'tinggi = paling sedikit dikeluhkan',
        'dimensi': dims,
    }

with open(OUT / 'puskesmas_profiles.json', 'w', encoding='utf-8') as f:
    json.dump(profiles, f, ensure_ascii=False, indent=1)
print(f'Tersimpan: puskesmas_profiles.json ({len(profiles)} puskesmas)')

In [8]:
# Topik kabupaten: sebaran tiap dimensi + spread antar-puskesmas + co-occurrence + chi2
topik = {}
neg = findings[(findings['polarity']=='neg') & findings['dimension'].isin(SERVQUAL_DIMS)]
total_neg_reviews = neg['review_id'].nunique()
for d in SERVQUAL_DIMS:
    sub = neg[neg['dimension']==d]
    pct = sub['review_id'].nunique() / total_neg_reviews
    # spread: di berapa puskesmas isu ini muncul (konsentrasi)
    n_pusk = sub['puskesmas_id'].nunique()
    topik[d] = {
        'persen_keluhan': round(pct*100, 1),
        'n_puskesmas_terdampak': int(n_pusk),
        'spread': 'merata' if n_pusk >= 0.7*findings['puskesmas_id'].nunique() else 'terkonsentrasi',
    }

dashboard_topik = {
    'topik': topik,
    'chi2_antar_wilayah': {'chi2': chi['chi2'], 'dof': chi['dof'], 'p': chi['p']},
    'residual_terstandarisasi': chi['std_residuals'].to_dict(),
    'co_occurrence_lift': lift.to_dict('records'),
}
with open(OUT / 'topik_kabupaten.json', 'w', encoding='utf-8') as f:
    json.dump(dashboard_topik, f, ensure_ascii=False, indent=1)
print('Tersimpan: topik_kabupaten.json')
print('\nRingkasan topik:')
for d, v in topik.items():
    print(f"  {d}: {v['persen_keluhan']}% keluhan, {v['n_puskesmas_terdampak']} puskesmas ({v['spread']})")

Tersimpan: topik_kabupaten.json

Ringkasan topik:
  Responsiveness: 42.1% keluhan, 126 puskesmas (merata)
  Reliability: 47.1% keluhan, 126 puskesmas (merata)
  Assurance: 20.6% keluhan, 124 puskesmas (merata)
  Empathy: 54.1% keluhan, 125 puskesmas (merata)
  Tangibles: 6.0% keluhan, 111 puskesmas (merata)


## 6. Ekspor hasil uji statistik (format readable untuk laporan)

Selain JSON dashboard, simpan tiap hasil analisis sebagai **CSV terpisah** + satu **Excel
multi-sheet** (`hasil_statistik.xlsx`) agar mudah dikutip di laporan esai. Disimpan di
`outputs/statistik/`.

In [ ]:
STAT = OUT / 'statistik'
STAT.mkdir(exist_ok=True)

# --- 1. Skor keluhan per puskesmas x dimensi (komposisi + intensitas + CI + skor) ---
skor_out = skor[['puskesmas_id','puskesmas_name','wilayah','dimension','n_reviews',
                 'n_complaint','komposisi_pct','intensitas_rate','intensitas_ci95',
                 'intensitas_shrunk','skor_1_5','kepercayaan','cukup_dinilai']].copy()
skor_out.columns = ['id_puskesmas','puskesmas','wilayah','dimensi','n_ulasan','n_keluhan',
                    'komposisi_%','intensitas','intensitas_CI95','intensitas_shrunk',
                    'skor_1_5','kepercayaan','cukup_dinilai']
skor_out.to_csv(STAT / 'skor_per_puskesmas.csv', index=False, encoding='utf-8-sig')

# --- 2. Badge perbandingan vs peer (uji dua proporsi) ---
peer_out = peer.copy()
peer_out.columns = ['id_puskesmas','dimensi','z','p_value','vs_peer','n_peer']
peer_out.to_csv(STAT / 'uji_perbandingan_peer.csv', index=False, encoding='utf-8-sig')

# mismatch bintang-teks
mismatch.to_csv(STAT / 'mismatch_bintang_teks.csv', index=False, encoding='utf-8-sig')

# --- 3a. Uji chi-square antar-wilayah: ringkasan + tabel + residual ---
chi_summary = pd.DataFrame([{
    'uji': 'Chi-square independensi (dimensi x wilayah)',
    'chi2': chi['chi2'], 'dof': chi['dof'], 'p_value': chi['p'],
    'kesimpulan': 'distribusi keluhan BERBEDA signifikan antar wilayah' if chi['p']<0.05
                  else 'tidak berbeda signifikan',
}])
chi_summary.to_csv(STAT / 'chi2_ringkasan.csv', index=False, encoding='utf-8-sig')
chi['contingency'].to_csv(STAT / 'chi2_tabel_kontingensi.csv', encoding='utf-8-sig')
chi['std_residuals'].to_csv(STAT / 'chi2_residual_terstandarisasi.csv', encoding='utf-8-sig')

# --- 3b. Co-occurrence lift antar-dimensi ---
lift_out = lift.copy()
lift_out.columns = ['dimensi_A','dimensi_B','P(A)','P(B)','P(A&B)','lift']
lift_out.to_csv(STAT / 'co_occurrence_lift.csv', index=False, encoding='utf-8-sig')

# --- 4. Daftar isu kanonik per dimensi (hasil clustering) + jumlah ulasan ---
isu_rows = []
for d in SERVQUAL_DIMS:
    c = canon_by_dim[d]
    g = c.groupby('cluster_label')['review_id'].nunique().sort_values(ascending=False)
    for isu, n in g.items():
        isu_rows.append({'dimensi': d, 'isu_kanonik': isu, 'n_ulasan': int(n)})
isu_df = pd.DataFrame(isu_rows)
isu_df.to_csv(STAT / 'isu_kanonik_per_dimensi.csv', index=False, encoding='utf-8-sig')

# --- 5. Sebaran topik se-kabupaten ---
topik_rows = [{'dimensi': d, 'persen_keluhan': topik[d]['persen_keluhan'],
               'n_puskesmas_terdampak': topik[d]['n_puskesmas_terdampak']} for d in SERVQUAL_DIMS]
pd.DataFrame(topik_rows).to_csv(STAT / 'sebaran_topik_kabupaten.csv', index=False, encoding='utf-8-sig')

# --- Gabung semua ke satu Excel multi-sheet (paling enak untuk laporan) ---
with pd.ExcelWriter(STAT / 'hasil_statistik.xlsx', engine='openpyxl') as xl:
    skor_out.to_excel(xl, sheet_name='1_skor_puskesmas', index=False)
    peer_out.to_excel(xl, sheet_name='2_uji_peer', index=False)
    mismatch.to_excel(xl, sheet_name='2_mismatch_bintang', index=False)
    chi_summary.to_excel(xl, sheet_name='3_chi2_ringkasan', index=False)
    chi['contingency'].to_excel(xl, sheet_name='3_chi2_kontingensi')
    chi['std_residuals'].to_excel(xl, sheet_name='3_chi2_residual')
    lift_out.to_excel(xl, sheet_name='3_lift', index=False)
    isu_df.to_excel(xl, sheet_name='4_isu_kanonik', index=False)
    pd.DataFrame(topik_rows).to_excel(xl, sheet_name='5_sebaran_topik', index=False)

print('Tersimpan di outputs/statistik/ :')
for f in sorted(STAT.glob('*')):
    print('  ', f.name)

## Selesai — output untuk dashboard & laporan

**Untuk dashboard:**
- `outputs/puskesmas_profiles.json` → tab Profil Puskesmas
- `outputs/topik_kabupaten.json` → tab Topik Keluhan

**Untuk laporan (readable):** `outputs/statistik/`
- `skor_per_puskesmas.csv` — skor + komposisi + intensitas + CI tiap puskesmas×dimensi
- `uji_perbandingan_peer.csv` — uji dua proporsi (z, p, verdict) + `mismatch_bintang_teks.csv`
- `chi2_ringkasan.csv` / `chi2_tabel_kontingensi.csv` / `chi2_residual_terstandarisasi.csv`
- `co_occurrence_lift.csv` — lift antar-dimensi
- `isu_kanonik_per_dimensi.csv` — hasil clustering + jumlah ulasan
- `sebaran_topik_kabupaten.csv`
- **`hasil_statistik.xlsx`** — semua di atas dalam satu file multi-sheet

**Metode statistik (untuk laporan esai):** proporsi + Wilson CI, empirical-Bayes Beta-Binomial
shrinkage, uji dua proporsi z, uji χ² independensi + residual terstandarisasi, lift
co-occurrence, embedding semantik + KMeans (silhouette) untuk kanonikalisasi isu.